# Underwater Image Enhancement — U-Net Training
**Environment:** Google Colab (T4 GPU)  
**Model:** U-Net with EfficientNetB0 pretrained encoder  
**Dataset:** UIEB (890 paired images)

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
# This connects Colab to your Google Drive so we can:
# 1. Read the dataset you uploaded there
# 2. Save trained model weights permanently
# Without this, everything is lost when the Colab session ends.

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

In [ ]:
# ── Clone GitHub Repo ─────────────────────────────────────────────────────────
# This pulls your src/ folder into Colab so we can import
# dataset.py, train.py etc. directly.

import subprocess

GITHUB_URL = "https://github.com/danielakbank/underwater-image-enhancement.git"

result = subprocess.run(
    ["git", "clone", GITHUB_URL, "/content/underwater"],
    capture_output=True,
    text=True
)

print(result.stdout or "✅ Repo cloned successfully")
print(result.stderr if result.returncode != 0 else "")

In [ ]:
# ── Install Dependencies ──────────────────────────────────────────────────────
# Colab already has TensorFlow, NumPy, and OpenCV pre-installed.
# We only install what's missing.

!pip install scikit-image tqdm --quiet

print("✅ Dependencies ready")

In [ ]:
# ── Install Dependencies ──────────────────────────────────────────────────────
# Colab already has TensorFlow, NumPy, and OpenCV pre-installed.
# We only install what's missing.

!pip install scikit-image tqdm --quiet

print("✅ Dependencies ready")

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────────────────────
# IMPORTANT: If this shows ❌, go to:
# Runtime → Change runtime type → Hardware accelerator → T4 GPU
# Then run all cells again from the top.

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"✅ GPU available: {gpus[0].name}")
    print(f"   TensorFlow: {tf.__version__}")
else:
    print("❌ No GPU — go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── Load Dataset ──────────────────────────────────────────────────────────────
# Reads images from your Google Drive folders.
# Make sure you have uploaded:
#   underwater-data/raw/        (890 degraded images)
#   underwater-data/reference/  (890 reference images)
# to your Google Drive before running this cell.

import sys
sys.path.insert(0, '/content/underwater/src')

from dataset import load_dataset

RAW_DIR = "/content/drive/MyDrive/underwater-data/raw"
REF_DIR = "/content/drive/MyDrive/underwater-data/reference"

print("Loading dataset — this may take 1–2 minutes...")
raw_images, ref_images = load_dataset(RAW_DIR, REF_DIR)

print(f"\n  Raw images shape  : {raw_images.shape}")
print(f"  Ref images shape  : {ref_images.shape}")
print(f"  Memory used       : {raw_images.nbytes / 1e6:.1f} MB")
print("  ✅ Dataset loaded")

In [ ]:
# ── Build Model and Data Pipelines ────────────────────────────────────────────
# Splits data into train/validation sets.
# Builds the U-Net model with pretrained EfficientNetB0 encoder.
# Builds optimised tf.data pipelines for GPU training.

import numpy as np
from train import build_model, build_dataset, build_callbacks, CONFIG

# ── Train / Validation split ──
val_size = int(len(raw_images) * CONFIG["val_split"])

train_raw = raw_images[val_size:]
train_ref = ref_images[val_size:]
val_raw   = raw_images[:val_size]
val_ref   = ref_images[:val_size]

print(f"  Training pairs   : {len(train_raw)}")
print(f"  Validation pairs : {len(val_raw)}")

# ── tf.data pipelines ──
# augment=True adds random horizontal flips to training data only
train_ds = build_dataset(train_raw, train_ref, augment=True)
val_ds   = build_dataset(val_raw,   val_ref,   augment=False)

# ── Build model ──
model = build_model(input_shape=(256, 256, 3))

print(f"\n  Total parameters : {model.count_params():,}")
print("  ✅ Model ready for training")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
# Trains the model and saves the best weights to Google Drive.
# Expected time on Colab T4 GPU: 20–40 minutes for 50 epochs.
#
# Watch for:
#   val_loss decreasing — model is learning
#   val_loss plateauing — ReduceLROnPlateau will kick in
#   EarlyStopping — training stops automatically if no improvement

import os

MODEL_SAVE_PATH = "/content/drive/MyDrive/underwater-data/unet_efficientnetb0.keras"
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

cb = build_callbacks(model_path=MODEL_SAVE_PATH)

print("Starting training...\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    callbacks=cb,
    verbose=1,
)

print(f"\n✅ Training complete")
print(f"   Best model saved to: {MODEL_SAVE_PATH}")

In [ ]:
# ── Training History ──────────────────────────────────────────────────────────
# Shows how training and validation loss changed across epochs.
# Both curves should decrease and level off together.
# If val_loss increases while train_loss decreases = overfitting.

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(history.history["loss"],     label="Train Loss", color="#e74c3c", linewidth=2)
ax.plot(history.history["val_loss"], label="Val Loss",   color="#3498db", linewidth=2)

ax.set_title("Training History — Loss per Epoch", fontsize=14, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(linestyle="--", alpha=0.5)

plt.tight_layout()

PLOT_PATH = "/content/drive/MyDrive/underwater-data/training_history.png"
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Training history saved to Drive")

In [ ]:
# ── Visual Results ────────────────────────────────────────────────────────────
# Runs 4 validation images through the trained model.
# Shows raw / CNN-enhanced / reference side by side.
# This is a quick sanity check — proper evaluation runs locally.

fig, axes = plt.subplots(4, 3, figsize=(14, 16))
col_titles = ["Raw (Degraded)", "CNN Enhanced", "Reference"]
col_colours = ["#e74c3c", "#2ecc71", "#3498db"]

for row in range(4):
    raw     = val_raw[row]
    ref     = val_ref[row]
    enhanced = model.predict(raw[np.newaxis, ...], verbose=0)[0]

    for col, (img, title, colour) in enumerate(
        zip([raw, enhanced, ref], col_titles, col_colours)
    ):
        axes[row, col].imshow(np.clip(img, 0, 1))
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(
                title, fontsize=12, fontweight="bold", color=colour
            )

plt.suptitle("CNN Enhancement — Validation Samples", fontsize=14, fontweight="bold")
plt.tight_layout()

SAMPLES_PATH = "/content/drive/MyDrive/underwater-data/cnn_samples.png"
plt.savefig(SAMPLES_PATH, dpi=150, bbox_inches="tight")
plt.show()

print("✅ Visual check complete")
print(f"   Saved to: {SAMPLES_PATH}")